## Setup — generate splits from Kaggle-extracted dataset
This cell builds `train.csv`/`test.csv` from `metadata.csv` if they don't already exist, matching the per-class Train/Test counts in `catalog_summary.csv`. Run this once before anything else.

In [ ]:
# --- Setup: generate train/test splits if missing (val.csv already provided) ---
import os
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_DIR = "/kaggle/working/dataset_extracted"
SPLITS_DIR = f"{DATA_DIR}/splits"

train_path = f"{SPLITS_DIR}/train.csv"
test_path  = f"{SPLITS_DIR}/test.csv"

if not (os.path.exists(train_path) and os.path.exists(test_path)):
    meta = pd.read_csv(f"{DATA_DIR}/metadata.csv")
    val  = pd.read_csv(f"{SPLITS_DIR}/val.csv")
    summary = pd.read_csv(f"{DATA_DIR}/catalog/catalog_summary.csv")

    remaining = meta[~meta["filename"].isin(val["filename"])].reset_index(drop=True)
    test_frac = summary["Test"].sum() / (summary["Train"].sum() + summary["Test"].sum())

    train_df, test_df = train_test_split(
        remaining,
        test_size=test_frac,
        stratify=remaining["unified_label"],
        random_state=42,
    )
    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path, index=False)
    print(f"Generated train.csv ({len(train_df)} rows) and test.csv ({len(test_df)} rows)")
else:
    print("train.csv and test.csv already exist, skipping generation")
